# Colab 3: Combined Two-Dataset Training

This notebook clones the repo, installs dependencies, finds two dataset folders by real CSV files, trains the combined model, evaluates it, zips the outputs, copies the zip to Drive, and downloads it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/au510621104021/FND_2027.git'
BRANCH = 'main'
PROJECT_DIR = '/content/FND_2027'

# Search roots for datasets. Add more paths here if needed.
DATASET_SEARCH_ROOTS = [
    '/content/drive/MyDrive',
    '/content'
]

# Name hints used when selecting the two datasets.
DATASET_1_HINTS = ['isot fake news dataset', 'isot']
DATASET_2_HINTS = ['fake news dataset']

# Leave as None to auto-discover. If auto-discovery still picks the wrong folders,
# paste the exact paths here and rerun from this cell.
DATASET_DIR_1 = None
DATASET_DIR_2 = None

DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/FND_2027_outputs'

In [ ]:
import os
from pathlib import Path

os.chdir('/content')
!rm -rf /content/FND_2027
!git clone --branch main https://github.com/au510621104021/FND_2027.git /content/FND_2027

assert Path(PROJECT_DIR).exists(), f'Project directory not found: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Project directory:', os.getcwd())
!ls

In [ ]:
import os
from pathlib import Path

os.chdir(PROJECT_DIR)
assert Path('requirements.txt').exists(), 'requirements.txt not found. Clone step likely failed.'
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

CSV_NAMES = {'train.csv', 'test.csv', 'dataset.csv', 'data.csv', 'Fake.csv', 'True.csv'}

def folder_has_dataset_csvs(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    names = {p.name for p in path.glob('*.csv')}
    return bool(names.intersection(CSV_NAMES))

def collect_dataset_candidates(search_roots):
    candidates = []
    seen = set()
    for root_str in search_roots:
        root = Path(root_str)
        if not root.exists():
            continue
        for csv_file in root.rglob('*.csv'):
            parent = csv_file.parent.resolve()
            if str(parent) in seen:
                continue
            if folder_has_dataset_csvs(parent):
                seen.add(str(parent))
                candidates.append(parent)
    return sorted(candidates, key=lambda p: str(p).lower())

def pick_candidate(candidates, hints):
    scored = []
    for path in candidates:
        text = str(path).lower()
        score = 0
        for hint in hints:
            if hint in path.name.lower():
                score += 5
            if hint in text:
                score += 2
        if score > 0:
            scored.append((score, len(path.parts), str(path)))
    if not scored:
        return None
    scored.sort(key=lambda x: (-x[0], x[1], x[2]))
    return scored[0][2]

candidates = collect_dataset_candidates(DATASET_SEARCH_ROOTS)
print('Detected dataset candidate folders:')
for candidate in candidates:
    print(' -', candidate)

if DATASET_DIR_1 is None:
    DATASET_DIR_1 = pick_candidate(candidates, DATASET_1_HINTS)
if DATASET_DIR_2 is None:
    DATASET_DIR_2 = pick_candidate(candidates, DATASET_2_HINTS)

print('\nSelected DATASET_DIR_1 =', DATASET_DIR_1)
print('Selected DATASET_DIR_2 =', DATASET_DIR_2)

assert DATASET_DIR_1 is not None, 'Could not find dataset 1 automatically. Use one of the printed candidate folders.'
assert DATASET_DIR_2 is not None, 'Could not find dataset 2 automatically. Use one of the printed candidate folders.'
assert Path(DATASET_DIR_1).exists(), f'Folder not found: {DATASET_DIR_1}'
assert Path(DATASET_DIR_2).exists(), f'Folder not found: {DATASET_DIR_2}'

print('\nDataset 1 CSVs:', [p.name for p in Path(DATASET_DIR_1).glob('*.csv')])
print('Dataset 2 CSVs:', [p.name for p in Path(DATASET_DIR_2).glob('*.csv')])

In [ ]:
import os
import yaml
from pathlib import Path

os.chdir(PROJECT_DIR)
config_path = Path('config/config.yaml')
with config_path.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config['data']['dataset_name'] = 'generic'
config['data']['data_dir'] = DATASET_DIR_1
config['data']['dataset_names'] = ['generic', 'generic']
config['data']['data_dirs'] = [DATASET_DIR_1, DATASET_DIR_2]
config['logging']['checkpoint_dir'] = './checkpoints'
config['logging']['log_dir'] = './logs'
config['inference']['model_checkpoint'] = './checkpoints/best_model_combined.pt'

with config_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print('Updated config for combined training:')
print(config['data'])

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python scripts/train.py --config config/config.yaml

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir(PROJECT_DIR)
best_model = Path('checkpoints/best_model.pt')
combined_model = Path('checkpoints/best_model_combined.pt')
assert best_model.exists(), 'Training did not produce checkpoints/best_model.pt'
shutil.copy2(best_model, combined_model)
print(f'Saved combined checkpoint to: {combined_model}')

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python scripts/evaluate.py --config config/config.yaml --checkpoint checkpoints/best_model_combined.pt

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir(PROJECT_DIR)
zip_base = Path('combined_model_artifacts')
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir='.', base_dir='checkpoints')
print(f'Created zip: {zip_path}')
!zip -ur combined_model_artifacts.zip results logs -x "*.ipynb_checkpoints*"

In [ ]:
import os
from pathlib import Path
from google.colab import files

os.chdir(PROJECT_DIR)
zip_path = Path('combined_model_artifacts.zip')
assert zip_path.exists(), 'Zip file was not created.'

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
!cp combined_model_artifacts.zip "$DRIVE_OUTPUT_DIR/"
print(f'Copied zip to Drive: {DRIVE_OUTPUT_DIR}')

files.download(str(zip_path))